<a href="https://colab.research.google.com/github/navitas26/navitas/blob/master/navitas-egitim-TEMIZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Navitas — Model Eğitimi (Colab / GPU) — Drive'SIZ sürüm

**Google Drive GEREKMEZ** (premium ödemen gerekmiyor). Veriler Colab'ın kendi diskine
iner, eğitilen model sonunda **GitHub'a push edilir** — bilgisayarında `git pull` ile alırsın.

**Nasıl:** Üstte **Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
Sonra her hücreyi sırayla **Shift+Enter** ile çalıştır.

Model: `yolov8n-seg`, 2 sınıf (`traversable`/`blocked`) · Veri: **RescueNet + FloodNet** · 100 epoch


## Adım 1 — GPU var mı?
Çıktıda bir GPU (ör. Tesla T4) görmelisin.

In [4]:
!nvidia-smi

Wed Jun 24 16:19:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Adım 2 — Projeyi GitHub'dan indir
Çalıştırınca **GitHub token** isteyecek (görünmez). Bu token hem indirmek hem de
sonunda modeli geri push etmek için kullanılır — `navitas26/navitas`'a erişimi olmalı.

In [4]:
from getpass import getpass
GITHUB_TOKEN = getpass("GitHub token yapıştır (görünmez): ")
REPO = "navitas26/navitas"
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git
!ls navitas

GitHub token yapıştır (görünmez): ··········
Cloning into 'navitas'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 132 (delta 2), reused 15 (delta 2), pack-reused 110 (from 1)
Receiving objects: 100% (132/132), 77.42 MiB | 31.29 MiB/s, done.
Resolving deltas: 100% (24/24), done.
codes	 doc	 navitas-egitim-TEMIZ.ipynb  simulation
dataset  models  README.md		     test_images


## Adım 3 — Gerekli kütüphaneleri kur

In [5]:
!pip install ultralytics datasets pillow tqdm opencv-python-headless -q
print('Kurulum tamam')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00
Kurulum tamam


## Adım 4 — Veri setlerini indir: RescueNet + FloodNet
Drive YOK; veriler Colab diskine iner (~10-20 dk). İki veri seti de iner.

In [6]:
%cd /content/navitas/dataset
print("=== RescueNet indiriliyor ===")
!python3 download_and_convert.py
print("=== FloodNet indiriliyor ===")
!python3 download_floodnet.py
print("Veri setleri hazır")

Streaming output truncated to the last 5000 lines.

8354.JPG.json: 100% 2.30k/2.30k [00:00<00:00, 9.41MB/s]

8368.JPG.json:   0% 0.00/2.30k [00:00<?, ?B/s]

8368.JPG.json: 100% 2.30k/2.30k [00:00<00:00, 1.82MB/s]
8369.JPG.json: 100% 2.28k/2.28k [00:00<00:00, 3.53MB/s]

8370.JPG.json: 100% 2.27k/2.27k [00:00<00:00, 3.71MB/s]

8374.JPG.json: 100% 2.29k/2.29k [00:00<00:00, 2.05MB/s]

8375.JPG.json: 100% 2.27k/2.27k [00:00<00:00, 9.31MB/s]

8379.JPG.json: 100% 580/580 [00:00<00:00, 3.53MB/s]

8381.JPG.json: 100% 584/584 [00:00<00:00, 1.56MB/s]

8382.JPG.json: 100% 584/584 [00:00<00:00, 2.84MB/s]

8380.JPG.json: 100% 584/584 [00:00<00:00, 3.33MB/s]

8366.JPG.json: 100% 1.43k/1.43k [00:00<00:00, 5.36MB/s]

8371.JPG.json: 100% 2.32k/2.32k [00:00<00:00, 7.98MB/s]

8385.JPG.json: 100% 584/584 [00:00<00:00, 1.98MB/s]

8361.JPG.json:   0% 0.00/1.44k [00:00<?, ?B/s]

8386.JPG.json: 100% 584/584 [00:00<00:00, 3.21MB/s]
8361.JPG.json: 100% 1.44k/1.44k [00:00<00:00, 1.63MB/s]
8387.JPG.json: 100% 584/

## Adım 5 — Eğitim ayar dosyasını yaz (HER İKİ veri seti dahil)

In [8]:
yaml_content = """train:
  - /content/navitas/dataset/rescuenet_yolo/images/train
  - /content/navitas/dataset/floodnet_yolo/images/train

val:
  - /content/navitas/dataset/rescuenet_yolo/images/val
  - /content/navitas/dataset/floodnet_yolo/images/val

test: /content/navitas/dataset/rescuenet_yolo/images/test

nc: 2
names:
  0: traversable
  1: blocked
"""
with open("/content/navitas/dataset/combined_data.yaml","w") as f:
    f.write(yaml_content)
print(yaml_content)
print("Ayar yazildi (RescueNet + FloodNet)")

train:
  - /content/navitas/dataset/rescuenet_yolo/images/train
  - /content/navitas/dataset/floodnet_yolo/images/train

val:
  - /content/navitas/dataset/rescuenet_yolo/images/val
  - /content/navitas/dataset/floodnet_yolo/images/val

test: /content/navitas/dataset/rescuenet_yolo/images/test

nc: 2
names:
  0: traversable
  1: blocked

Ayar yazildi (RescueNet + FloodNet)


In [9]:
!ls /content
!echo "--- navitas içi ---"; ls /content/navitas 2>&1 | head
!ls -la /content/navitas/codes/train_seg.py 2>&1

navitas  sample_data
--- navitas içi ---
codes
dataset
doc
models
navitas-egitim-TEMIZ.ipynb
README.md
simulation
test_images
-rw-r--r-- 1 root root 7901 Jun 24 17:16 /content/navitas/codes/train_seg.py


## Adım 6 — GPU için eğitim ayarı (device=0, batch=16)

In [10]:
!sed -i 's/device="cpu"/device=0/' /content/navitas/codes/train_seg.py
!sed -i 's/batch=8,/batch=16,/' /content/navitas/codes/train_seg.py
!grep -nE "device=|batch=" /content/navitas/codes/train_seg.py

40:        batch=16,                # CPU için optimum
42:        device=0,


## Adım 7 — Eğit (100 epoch · ~1-2 saat)
Uzun sürer; sekmeyi kapatma.

In [11]:
%cd /content/navitas/codes
!python3 train_seg.py --data /content/navitas/dataset/combined_data.yaml --epochs 100

/content/navitas/codes
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model   : yolov8n-seg.pt
Dataset : /content/navitas/dataset/combined_data.yaml
Epochs  : 100  (patience=40)
Run     : navitas_seg_v2
Device  : CPU (AMD Ryzen 7 5700U, 16 çekirdek)
Ultralytics 8.4.76 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=25, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/navitas/dataset/combined_data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropo

## Adım 8 — Eğitilmiş modeli GitHub'a PUSH et
Model dosyalarını repoya ekleyip `navitas26/navitas`'a gönderir.
Bilgisayarında `git pull` yapınca model gelir. (Drive'a gerek yok.)

In [12]:
import glob, shutil, os
os.chdir('/content/navitas')
hits = sorted(glob.glob('codes/runs/segment/**/weights/best.pt', recursive=True))
print('Bulunan best.pt:', hits)
os.makedirs('models', exist_ok=True)
shutil.copy(hits[0], 'models/navitas_seg_v2_best.pt')
print('best.pt -> models/navitas_seg_v2_best.pt')

!git config user.email "hattab1342@gmail.com"
!git config user.name "navitas-colab"
!git add -f models/navitas_seg_v2_best.pt
!git add -f models/navitas_seg_v2_ncnn_model
!git commit -m "model: navitas_seg_v2 (Colab egitim, RescueNet+FloodNet, 100ep) v24.06 1"
!git pull --no-edit origin master
!git push origin HEAD:master
print(">>> Bitti. Hata yoksa model GitHub'da. Bilgisayarinda: git pull")

Bulunan best.pt: ['codes/runs/segment/runs/segment/navitas_seg_v2/weights/best.pt']
best.pt -> models/navitas_seg_v2_best.pt
[master 594e1b0] model: navitas_seg_v2 (Colab egitim, RescueNet+FloodNet, 100ep) v24.06 1
 6 files changed, 278 insertions(+)
 create mode 100644 models/navitas_seg_v2_best.pt
 create mode 100644 models/navitas_seg_v2_ncnn_model/__pycache__/model_pnnx.cpython-312.pyc
 create mode 100644 models/navitas_seg_v2_ncnn_model/metadata.yaml
 create mode 100644 models/navitas_seg_v2_ncnn_model/model.ncnn.bin
 create mode 100644 models/navitas_seg_v2_ncnn_model/model.ncnn.param
 create mode 100644 models/navitas_seg_v2_ncnn_model/model_ncnn.py
From https://github.com/navitas26/navitas
 * branch            master     -> FETCH_HEAD
Already up to date.
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (11/11), 16.75 MiB | 8.33 MiB/s, done.
Total 11 (delta 

## Adım 9 (YEDEK) — Push olmazsa modeli doğrudan indir
Adım 8'de push hata verirse bunu çalıştır; model zip olarak bilgisayarına iner.

In [13]:
import shutil
from google.colab import files
shutil.copytree('/content/navitas/models/navitas_seg_v2_ncnn_model',
                '/content/cikti/navitas_seg_v2_ncnn_model', dirs_exist_ok=True)
shutil.copy('/content/navitas/models/navitas_seg_v2_best.pt',
            '/content/cikti/navitas_seg_v2_best.pt')
shutil.make_archive('/content/navitas_model', 'zip', '/content/cikti')
files.download('/content/navitas_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>